# Remove fixed pattern from spectrograph images
There is a constant pattern present throughout the FDM. Take pixels from off the disk to generate this fixed pattern image and remove it from the spectrograph data. Input Level 1.1 data.
The next step after this is `despike_and_save.ipynb`.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
# %matplotlib notebook

import pathlib as pl
import numpy as np
from skimage.filters import threshold_otsu
import pickle
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.pyplot as plt
from matplotlib import colors
# params = {"ytick.color" : "k",
#           "xtick.color" : "k",
#           "axes.labelcolor" : "k",
#           "axes.edgecolor" : "k"}
# plt.rcParams.update(params)
import astropy.units as u
from astropy.nddata import block_reduce

from iris_mosaics import read_sg_image, build_mosaic_single_wavelength
import iris_mosaics as iris_fdm
from astropy.modeling import models, fitting
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support
quantity_support()

Load spectrograph images

In [ ]:
# Paths of spectrograph images
path = pl.Path(r'D:\IRIS data\deep_mosaics\20240811')
# path_level_11 = path / 'level_11'
path_level_11 = path / 'level_11_plus_iris_prep_bg_sub'

files = list(path_level_11.glob('*.fits'))
# files = files[:1000]

# Length of number of images dimension
num_imgs = len(files)

Examine iris_prep.pro images with and without dark subtraction

In [ ]:
# w, hdu, _ = read_sg_image(path / 'no_dark_test.fits','fuv2')
# img_no_dark = hdu[0].data
#
# w, hdu, _ = read_sg_image(path / 'dark_test.fits','fuv2')
# img_dark = hdu[0].data

In [ ]:
# w, hdu, _ = read_sg_image(path / 'no_flat_test.fits','fuv2')
# img_no_flat = hdu[0].data
#
# w, hdu, _ = read_sg_image(path / 'flat_test.fits','fuv2')
# img_flat = hdu[0].data

In [ ]:
# plt.figure(figsize=(15,5))
# plt.imshow(img_no_dark - img_dark,
#            vmax=150,
#            vmin=50
#            )
# plt.colorbar().set_label('DN')

In [ ]:
# plt.figure(figsize=(15,5))
# plt.imshow(img_no_flat - img_flat,
#            vmax=.5,
#            vmin=-.5
#            )
# plt.colorbar().set_label('DN')

### This cell should be used later in the pipeline to rebin the data after iris_prep block #2.

In [ ]:
def nansum_keepnan(a, axis):
    s = np.nansum(a, axis=axis)
    s[np.all(np.isnan(a), axis=axis)] = np.nan   # all-NaN block -> stay NaN
    return s

# Load images
sg_img = np.empty((num_imgs, 548, 1036))
sg_wcs = []

for i, file in enumerate(files):
    w, hdu, _ = read_sg_image(file, 'fuv2')
    img = hdu[0].data
    sg_img[i] = block_reduce(img, (1, 2), func=nansum_keepnan)
    sg_wcs.append(w)

### Use this cell to deal with the 2x spectrally binned data that is 2072 pix wide.
This cell reads in only the second half of the CCD (where the Si IV lines are), so the data is easier to work with for now.

In [ ]:
%%time
# Load images
sg_img = np.empty((num_imgs, 548, 1036))
sg_wcs = []

for i, file in enumerate(files):
    w, hdu, _ = read_sg_image(file, 'fuv2')
    img = hdu[0].data[...,1036:]
    sg_img[i] = img
    sg_wcs.append(w)

In [ ]:
hdr = hdu[0].header
hdr

In [ ]:
%%time
# Load images
sg_wcs = []
sg_img = []

for i, file in enumerate(files):
    
    w, hdu, _ = read_sg_image(file,'fuv2')
    img = hdu[0].data

    sg_wcs.append(w)
    sg_img.append(img)

sg_img = np.array(sg_img)

Retain image of data before correction

In [ ]:
sg_img[sg_img == np.inf] = np.nan

In [ ]:
sg_img_mean = np.nanmean(sg_img, axis=0)

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(sg_img_mean,
           vmax=np.nanpercentile(sg_img_mean,99.9),
           vmin=0,
           origin='lower',)
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

plt.savefig(path / 'sg_img_11_mean.png',dpi=300, transparent=True, bbox_inches='tight')

with open(path / 'sg_img_11_mean.pickle', 'wb') as fh:
    pickle.dump(sg_img_mean, fh)

In [ ]:
# plt.figure(figsize=(15,5))
# plt.imshow(sg_img_mean,
#            vmax=80,
#            vmin=0)
# plt.colorbar().set_label('DN')
# .savefig('original_may_2025.png',dpi=300, transparent=True)

Wavelength array

In [ ]:
# For normal mosaics...
# sg_wavelength_full = sg_wcs[0].array_index_to_world(*np.indices((1, 1, sg_img[0].shape[1])))[0].to(u.Angstrom)

# For mosaics with twice as many pixels in the x direction (2072 vs 1036)...
sg_wavelength_full = sg_wcs[0].array_index_to_world(*np.indices((1, 1, 2*sg_img[0].shape[1])))[0].to(u.Angstrom)
sg_wavelength_half = sg_wavelength_full[..., 1036:]

Function to identify off-disk pixels

In [ ]:
def is_off_disk(x: u.Quantity, y: u.Quantity):

    # NOTE: next time you run this, see what this value looks like:
    # hdu[0].header['RSUN_OBS']

    # 20190912
    # limb_radius = 965 * u.arcsec

    # 20140324
    # limb_radius = 970 * u.arcsec

    # 20240811
    limb_radius = hdu[0].header['RSUN_OBS'] * u.arcsec + 10 * u.arcsec

    # ***Initially set to zero to see how much x and y offset we need***
    # off_disk_dr = 0 * u.arcsec

    # To minimize stray light, define off-disk pixels as those that are XX arcsec away
    off_disk_dr = 60 * u.arcsec

    # Total radius of occulting disk
    radius_occulting_disk = limb_radius + off_disk_dr

    # Seemed to need a shift to center the circle mask on the solar disk...
    # 20290912
    # offset_x = -1 * u.arcsec
    # offset_y = 12 * u.arcsec

    # 20140324
    # offset_x = 0 * u.arcsec
    # offset_y = 5 * u.arcsec

    # 20240811
    offset_x = -4 * u.arcsec
    offset_y = 10 * u.arcsec

    x += offset_x
    y += offset_y

    where_off_disk = np.square(x) + np.square(y) > np.square(radius_occulting_disk)
    # where_up = y > 0

    return where_off_disk

Check that disk is centered; modify offsets if needed

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(sg_img_mean,
           vmax=np.nanpercentile(sg_img_mean,99.9),
           vmin=0,
           origin='lower',)
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])
plt.axvline(520)

In [ ]:
%%time
# Pick a pixel slice through the Si IV 1394 line for plotting purposes. NOTE: Not geometrically corrected

# For normal mosaics...
# wav_sl = 775

# For ones with twice as many pixels in the x direction...
wav_sl = 520

disk_mask = np.empty(sg_img[...,wav_sl:wav_sl+1].shape, dtype=bool)
indices_sg_img = np.indices(sg_img[0,:,wav_sl:wav_sl+1].shape)

# Identify off-disk pixels
for i, img in enumerate(sg_img[...,wav_sl:wav_sl+1]):
    world_coord_indices = sg_wcs[i].array_index_to_world(0, *indices_sg_img)
    wavelength, y, x = world_coord_indices
    mask = abs(x) > 180 * u.deg
    x[mask] = x[mask] % (360 * u.deg)
    disk_mask[i] = is_off_disk(x, y)

# Set masked pixels to NaNs
sg_off_disk = sg_img[...,wav_sl:wav_sl+1].copy()
sg_off_disk[~disk_mask.astype(bool)] = np.nan

# Build mosaic
global_data, global_wcs = build_mosaic_single_wavelength(sg_off_disk, sg_wcs)

# Plot mosaic
plt.figure(figsize=(15,15))
plt.imshow(np.squeeze(global_data).T, aspect=global_data.shape[0]/global_data.shape[1], vmax=60)
plt.colorbar()

In [ ]:
%%time
# Set masked pixels to NaNs
sg_on_disk = sg_img[...,wav_sl:wav_sl+1].copy()
sg_on_disk[disk_mask.astype(bool)] = np.nan

# Build mosaic
global_data_on_disk, global_wcs_on_disk = build_mosaic_single_wavelength(sg_on_disk, sg_wcs)

# Plot mosaic
plt.figure(figsize=(15,15))
plt.imshow(np.squeeze(global_data_on_disk).T, aspect=global_data_on_disk.shape[0]/global_data_on_disk.shape[1], vmax=80)
plt.colorbar()

Apply to spectrograph images

In [ ]:
%%time
disk_mask = np.empty(sg_img.shape, dtype=bool)
indices_sg_img = np.indices(sg_img[0].shape)

for i, img in enumerate(sg_img):
    world_coord_indices = sg_wcs[i].array_index_to_world(0, *indices_sg_img)
    wavelength, y, x = world_coord_indices
    mask = abs(x) > 180 * u.deg
    x[mask] = x[mask] % (360 * u.deg)
    disk_mask[i] = is_off_disk(x, y)

In [ ]:
disk_mask_sum = np.sum(disk_mask, axis=0)

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(disk_mask_sum, origin='lower')
plt.colorbar().set_label('no. of data points')
plt.title('Number of off-disk data points we have for each pixel in the CCD')

Define off-disk array and set pixels inside the disk to NaN

In [ ]:
%%time
sg_off_disk = sg_img.copy()
sg_off_disk[~disk_mask.astype(bool)] = np.nan

Plot sample to show what data will be used for off-disk image

In [ ]:
fdm_range_start = 10000
fdm_range_end = 11000

plt.figure(figsize=(5,7))
plt.imshow(disk_mask[fdm_range_start:fdm_range_end,:,wav_sl],
           # vmin=-5,
           # vmax=20
           )
# plt.title('off-disk data')
plt.colorbar()
# plt.savefig('off_disk_mask_lmsal.png',dpi=300, transparent=True)

plt.figure(figsize=(5,7))
plt.imshow(sg_img[fdm_range_start:fdm_range_end,:,wav_sl] * ~disk_mask[fdm_range_start:fdm_range_end,:,wav_sl],
           # vmin=-5,
           vmax=100
           )
# plt.title('spectrograph data')
plt.colorbar()
# plt.savefig('sample_disk_lmsal.png',dpi=300, transparent=True)

Claude-coded cell to apply nanmean on the whole dataset without running out of memory.

In [ ]:
# Kept running out of memory trying to generate the nanmean image... this does it in "chunks"
shape = sg_off_disk.shape[1:]                    # (548, 1036)
acc = np.zeros(shape, dtype=np.float64)
cnt = np.zeros(shape, dtype=np.int64)

chunk = 500                                  # frames per chunk; tune to taste
for i0 in range(0, sg_off_disk.shape[0], chunk):
    block = sg_off_disk[i0:i0 + chunk]
    good = ~np.isnan(block)
    acc += np.where(good, block, 0).sum(axis=0, dtype=np.float64)
    cnt += good.sum(axis=0)

sg_off_disk_mean = np.divide(acc, cnt, out=np.full(shape, np.nan), where=cnt > 0)

In [ ]:
sg_off_disk_mean = np.nanmean(sg_off_disk, axis=0)

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(sg_off_disk_mean,
           vmin=0,
           vmax=np.nanpercentile(sg_off_disk_mean,99.8),
           origin='lower'
           )
# plt.xlim(723, 1035)
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

plt.savefig(path / 'sg_img_11_off_disk_mean.png',dpi=300, transparent=True, bbox_inches='tight')

with open(path / 'sg_img_11_off_disk_mean.pickle', 'wb') as fh:
    pickle.dump(sg_off_disk_mean, fh)

In [ ]:
# plt.figure(figsize=(15,5))
# plt.imshow(sg_off_disk_mean,
#            vmin=0,
#            vmax=50
#            )
# # plt.xlim(723, 1035)
# plt.colorbar().set_label('DN')

Make a histogram of the off disk values

In [ ]:
plt.figure(figsize=(10,5))
alpha = 1
num_bins = 250
hist1, edges1 = np.histogram(sg_off_disk[np.isfinite(sg_off_disk)], bins=num_bins)
plt.stairs(hist1, edges1, fill=False, alpha = alpha)
plt.xlabel('DN')
plt.ylabel('occurrences')
plt.yscale('log')

In [ ]:
# plt.figure(figsize=(10,5))
# alpha = 1
# num_bins = 250
# hist1, edges1 = np.histogram(sg_off_disk[np.isfinite(sg_off_disk)], bins=num_bins)
# plt.stairs(hist1, edges1, fill=False, alpha = alpha)
# plt.xlabel('DN')
# plt.ylabel('occurrences')
# plt.yscale('log')
# plt.xscale('log')
# plt.xlim((-10,1000))

In [ ]:
plt.figure(figsize=(10,5))
alpha = 1
num_bins = 6000
hist1, edges1 = np.histogram(sg_off_disk[np.isfinite(sg_off_disk)], bins=num_bins)
plt.stairs(hist1, edges1, fill=False, alpha = alpha)
plt.xlabel('DN')
plt.ylabel('occurrences')
plt.yscale('log')
plt.xlim((-25,200))

In [ ]:
# plt.figure(figsize=(10,5))
# alpha = 1
# num_bins = 6000
# hist1, edges1 = np.histogram(sg_off_disk[np.isfinite(sg_off_disk)], bins=num_bins)
# plt.stairs(hist1, edges1, fill=False, alpha = alpha)
# plt.xlabel('DN')
# plt.ylabel('occurrences')
# plt.yscale('log')
# plt.xlim((-25,200))

#### Cut top 10% to ensure the majority of the spikes are removed
If it's not cut, Otsu's method doesn't work later on.

In [ ]:
%%time
percent_to_cut = 10
upper_threshold = np.nanpercentile(sg_off_disk, 100 - percent_to_cut, axis=0)
sg_off_disk[sg_off_disk > upper_threshold] = np.nan

In [ ]:
# If this fails due to memory, use the next cell.
sg_off_disk_trimmed_mean = np.nanmean(sg_off_disk, axis=0)

In [ ]:
# Kept running out of memory trying to generate the nanmean image... this does it in "chunks"
shape = sg_off_disk.shape[1:]                    # (548, 1036)
acc = np.zeros(shape, dtype=np.float64)
cnt = np.zeros(shape, dtype=np.int64)

chunk = 500                                  # frames per chunk; tune to taste
for i0 in range(0, sg_off_disk.shape[0], chunk):
    block = sg_off_disk[i0:i0 + chunk]
    good = ~np.isnan(block)
    acc += np.where(good, block, 0).sum(axis=0, dtype=np.float64)
    cnt += good.sum(axis=0)

sg_off_disk_trimmed_mean = np.divide(acc, cnt, out=np.full(shape, np.nan), where=cnt > 0)

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(sg_off_disk_trimmed_mean,
           vmin=-2,
           vmax=np.nanpercentile(sg_off_disk_trimmed_mean,99.9),
           # norm=colors.PowerNorm(.5, vmin=-2, vmax=np.nanpercentile(sg_off_disk_trimmed_mean,99.99)),
           origin='lower'
           )
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

plt.savefig(path / 'sg_img_11_off_disk_one_sided_trimmed_mean.png',dpi=300, transparent=True, bbox_inches='tight')

with open(path / 'sg_img_11_off_disk_one_sided_trimmed_mean.pickle', 'wb') as fh:
    pickle.dump(sg_off_disk_trimmed_mean, fh)

In [ ]:
# plt.figure(figsize=(15,5))
# plt.imshow(sg_off_disk_trimmed_mean,
#            vmin=0,
#            vmax=40
#            )
# # plt.xlim(723, 1035)
# plt.colorbar().set_label('DN')

Subtract fixed pattern image from dataset

In [ ]:
sg_img -= sg_off_disk_trimmed_mean

Mean spectrograph image after subtracting fixed pattern image

In [ ]:
sg_img_mean_corrected = np.nanmean(sg_img, axis=0)

In [ ]:
# Kept running out of memory trying to generate the nanmean image... this does it in "chunks"
shape = sg_img.shape[1:]                    # (548, 1036)
acc = np.zeros(shape, dtype=np.float64)
cnt = np.zeros(shape, dtype=np.int64)

chunk = 500                                  # frames per chunk; tune to taste
for i0 in range(0, sg_img.shape[0], chunk):
    block = sg_img[i0:i0 + chunk]
    good = ~np.isnan(block)
    acc += np.where(good, block, 0).sum(axis=0, dtype=np.float64)
    cnt += good.sum(axis=0)

sg_img_mean_corrected = np.divide(acc, cnt, out=np.full(shape, np.nan), where=cnt > 0)

Mean images before and after

In [ ]:
plot_min = 0

plt.figure(figsize=(15,5))
plt.imshow(sg_img_mean,
           # vmin=plot_min,
           # vmax=np.nanpercentile(sg_img_mean_corrected,99.99),
           # vmax=np.nanpercentile(sg_img_mean_corrected,99.0),
           norm=colors.PowerNorm(.5, vmin=plot_min, vmax=np.nanpercentile(sg_img_mean,99.99)),
           origin='lower'
           )
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

plt.figure(figsize=(15,5))
plt.imshow(sg_img_mean_corrected,
           # vmin=plot_min,
           # vmax=np.nanpercentile(sg_img_mean_corrected,99.99),
           # vmax=np.nanpercentile(sg_img_mean_corrected,99.0),
           norm=colors.PowerNorm(.5, vmin=plot_min, vmax=np.nanpercentile(sg_img_mean_corrected,99.99)),
           origin='lower'
           )
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

plt.savefig(path / 'sg_img_11_mean_minus_off_disk_trimmed_mean.png',dpi=300, transparent=True, bbox_inches='tight')

with open(path / 'sg_img_11_mean_minus_off_disk_trimmed_mean.pickle', 'wb') as fh:
    pickle.dump(sg_img_mean_corrected, fh)

In [ ]:
# plot_min = 0
# plot_max = 70
#
# plt.figure(figsize=(15,5))
# plt.imshow(sg_img_mean,
#            vmin=plot_min,
#            vmax=plot_max
#            )
# plt.colorbar().set_label('DN')
#
# plt.figure(figsize=(15,5))
# plt.imshow(sg_img_mean_corrected,
#            vmin=plot_min,
#            vmax=plot_max
#            )
# plt.colorbar().set_label('DN')
# plt.savefig('mean_fixed_pattern_removed_level_12_lmsal.png',dpi=300, transparent=True)

Spectra before and after removing fixed pattern

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(np.nanmean(sg_img_mean, axis=0), label='original')
plt.plot(np.nanmean(sg_off_disk_mean, axis=0), label='off-disk')
plt.plot(np.nanmean(sg_img_mean_corrected, axis=0), label='corrected')
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
plt.legend()
plt.ylim((-2,15))

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(np.nanmean(sg_img_mean, axis=0) - np.nanmean(sg_img_mean), label='original')
# plt.plot(np.nanmean(sg_off_disk_mean, axis=0) - np.nanmean(sg_off_disk_mean), label='off-disk')
# plt.plot(np.nanmean(sg_img_mean_corrected, axis=0) - np.nanmean(sg_img_mean_corrected), label='corrected')
# plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
# plt.legend()
# plt.savefig('disk_spectrum_fixed_pattern_removed.pdf', dpi=300, transparent=True)

Save data

In [ ]:
for (file, corrected_image) in zip(files, sg_img):
    w, hdu, _ = read_sg_image(file,'fuv2')
    hdu[0].data = corrected_image
    # new_file = file.parent.parent / 'level_11_full_ccd_fixed_pattern_removed' / file.name
    new_file = file.parent.parent / 'level_11_iris_prep_bgsub_fixed_pattern_removed' / file.name
    hdu.writeto(new_file, overwrite=True)

If you only read in the FUV-L CCD, use this cell to replace the old FUV-L with the fix-pattern-removed one and save the images.

In [ ]:
sl_fuv_l = slice(None), slice(1036,None)

for i, (file, corrected_image) in enumerate(zip(files, sg_img)):
    wcs, hdul, _ = read_sg_image(file,'fuv2')
    sg_image = hdul[0].data
    sg_image[sl_fuv_l] = corrected_image
    hdul[0].data = sg_image
    new_file = file.parent.parent / 'level_11_iris_prep_bgsub_fixed_pattern_removed' / file.name
    hdul.writeto(new_file, overwrite=True)